# 4. Hypothesis Testing: t-test, Chi-Square, ANOVA
Works on **any dataset**. A **file explorer window** opens in Step 2. Columns for each test are **auto-selected** (printed in Step 3) - override them in the CONFIG cell if your question needs specific columns.

**Tests covered:** One-sample t-test | Independent two-sample t-test (Student / Welch) | Paired t-test | Chi-square test of independence | Chi-square goodness of fit | One-way ANOVA + Tukey post-hoc | non-parametric alternatives (Mann-Whitney, Wilcoxon, Kruskal-Wallis, Fisher)

## Step 0: Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)

## Step 1: CONFIG (leave as None for automatic selection)

In [ ]:
ALPHA        = 0.05          # significance level
ALTERNATIVE  = 'two-sided'   # for t-tests: 'two-sided' | 'greater' | 'less'

# ---- One-sample t-test ----
T1_COL   = None              # numeric column, e.g. 'age'
MU0      = None              # hypothesised population mean from your question, e.g. 35

# ---- Independent two-sample t-test ----
T2_NUM    = None             # numeric column, e.g. 'salary'
T2_GROUP  = None             # categorical column, e.g. 'gender'
T2_LEVELS = None             # two group names, e.g. ('Male', 'Female'). None = automatic

# ---- Paired t-test (before/after style columns) ----
PAIR_COL1 = None             # e.g. 'before'
PAIR_COL2 = None             # e.g. 'after'

# ---- Chi-square test of independence ----
CHI_COL1 = None              # categorical column 1
CHI_COL2 = None              # categorical column 2

# ---- Chi-square goodness of fit ----
GOF_COL   = None             # categorical column
GOF_PROPS = None             # expected proportions, e.g. [0.5, 0.3, 0.2] in sorted-category order. None = all equal

# ---- One-way ANOVA ----
ANOVA_NUM   = None           # numeric column
ANOVA_GROUP = None           # categorical column with 3+ groups

## Step 2: Select Dataset from File Explorer

In [ ]:
def pick_file():
    """Opens the OS file explorer. Falls back to typing the path if no GUI is available."""
    try:
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk()
        root.withdraw()
        root.attributes('-topmost', True)
        path = filedialog.askopenfilename(
            title='Select your dataset',
            filetypes=[('Data files', '*.csv *.xlsx *.xls *.json *.txt *.tsv'), ('All files', '*.*')])
        root.destroy()
    except Exception:
        print('File dialog not available -> type the path instead.')
        path = input('Enter full file path: ')
    return path.strip().strip('"').strip("'")

def load_data(path):
    ext = os.path.splitext(path)[1].lower()
    if ext in ('.xlsx', '.xls'):
        return pd.read_excel(path)
    if ext == '.json':
        return pd.read_json(path)
    try:
        return pd.read_csv(path, sep=None, engine='python')
    except UnicodeDecodeError:
        return pd.read_csv(path, sep=None, engine='python', encoding='latin-1')

file_path = pick_file()
if not file_path:
    raise SystemExit('No file selected. Re-run this cell.')

df = load_data(file_path)
df.columns = df.columns.astype(str).str.strip()
print('Loaded:', file_path, '| Shape:', df.shape)
df.head()

## Step 3: Light Cleaning + Automatic Column Selection

In [ ]:
def text_cols(d):
    return [col for col in d.columns
            if pd.api.types.is_object_dtype(d[col]) or pd.api.types.is_string_dtype(d[col])]

for col in text_cols(df):
    df[col] = df[col].where(df[col].isna(), df[col].astype(str).str.strip())
df = df.replace(['?', 'NA', 'N/A', 'na', 'n/a', 'NaN', 'nan', 'null', 'NULL', 'None', '-', '--', ''], np.nan)
for col in text_cols(df):                                  # numbers stored as text -> numeric
    conv = pd.to_numeric(df[col], errors='coerce')
    if df[col].notna().sum() > 0 and conv.notna().sum() >= 0.9 * df[col].notna().sum():
        df[col] = conv
for col in df.select_dtypes(include='bool').columns:       # booleans -> text categories
    df[col] = df[col].astype(str)

txt       = text_cols(df)
num_cols  = df.select_dtypes(include=np.number).columns.tolist()
cont_cols = [col for col in num_cols if df[col].nunique() > 10] or num_cols          # continuous numeric
cat_cols  = [col for col in df.columns
             if 2 <= df[col].nunique() <= 10 and (col in txt or col not in cont_cols)]  # categorical candidates
two_lvl   = [col for col in cat_cols if df[col].nunique() == 2]
multi_lvl = [col for col in cat_cols if df[col].nunique() >= 3]

def pick(user, default, name):
    if user is None:
        return default
    if user not in df.columns:
        raise KeyError(f"{name}: column '{user}' not found. Available: {list(df.columns)}")
    return user

s_t1_col  = pick(T1_COL,  cont_cols[0] if cont_cols else None, 'T1_COL')
s_t2_num  = pick(T2_NUM,  cont_cols[0] if cont_cols else None, 'T2_NUM')
s_t2_grp  = pick(T2_GROUP, (two_lvl or cat_cols or [None])[0], 'T2_GROUP')
s_pair1   = pick(PAIR_COL1, cont_cols[0] if len(cont_cols) >= 2 else None, 'PAIR_COL1')
s_pair2   = pick(PAIR_COL2, cont_cols[1] if len(cont_cols) >= 2 else None, 'PAIR_COL2')
s_chi1    = pick(CHI_COL1, cat_cols[0] if len(cat_cols) >= 2 else None, 'CHI_COL1')
s_chi2    = pick(CHI_COL2, cat_cols[1] if len(cat_cols) >= 2 else None, 'CHI_COL2')
s_gof     = pick(GOF_COL,  cat_cols[0] if cat_cols else None, 'GOF_COL')
s_an_num  = pick(ANOVA_NUM, cont_cols[0] if cont_cols else None, 'ANOVA_NUM')
s_an_grp  = pick(ANOVA_GROUP, (multi_lvl or cat_cols or [None])[0], 'ANOVA_GROUP')

print('Continuous numeric columns:', cont_cols)
print('Categorical columns       :', cat_cols)
print('\nAuto-selected columns (change them in CONFIG if needed):')
print(f'  One-sample t-test    : {s_t1_col}')
print(f'  Two-sample t-test    : {s_t2_num} by {s_t2_grp}')
print(f'  Paired t-test        : {s_pair1} vs {s_pair2}')
print(f'  Chi-square independ. : {s_chi1} x {s_chi2}')
print(f'  Chi-square GOF       : {s_gof}')
print(f'  One-way ANOVA        : {s_an_num} by {s_an_grp}')

## Helpers: decision printer, normality & variance checks

In [ ]:
summary = []      # every test result is stored here and shown at the end

def decide(test, variables, stat_name, stat, p, h0, h1):
    reject = p <= ALPHA
    print('-' * 70)
    print(f'H0: {h0}')
    print(f'H1: {h1}')
    print(f'Significance level (alpha) = {ALPHA}')
    print(f'{stat_name} = {stat:.4f}')
    print(f'p-value = {p:.6f}')
    if reject:
        print(f'DECISION: p ({p:.4f}) <= alpha ({ALPHA})  ->  REJECT H0  (statistically significant)')
    else:
        print(f'DECISION: p ({p:.4f}) > alpha ({ALPHA})  ->  FAIL TO REJECT H0  (not significant)')
    print('-' * 70)
    summary.append({'Test': test, 'Variables': variables, 'Statistic': round(float(stat), 4),
                    'p-value': round(float(p), 6), 'Decision': 'Reject H0' if reject else 'Fail to reject H0'})

def shapiro_p(arr):
    arr = np.asarray(arr, dtype=float)
    if len(arr) < 3 or np.ptp(arr) == 0:
        return np.nan
    if len(arr) > 5000:
        arr = np.random.default_rng(0).choice(arr, 5000, replace=False)
    return stats.shapiro(arr).pvalue

def check_assumptions(groups):
    """groups: dict name -> array. Prints Shapiro-Wilk (normality) and Levene (equal variances)."""
    rows = [[g, len(v), round(np.mean(v), 3), round(np.std(v, ddof=1), 3), shapiro_p(v)] for g, v in groups.items()]
    tab = pd.DataFrame(rows, columns=['group', 'n', 'mean', 'std', 'Shapiro p']).round(4)
    tab['normal?'] = np.where(tab['Shapiro p'] > ALPHA, 'Yes', 'No')
    display(tab)
    lev = stats.levene(*groups.values())
    equal_var = lev.pvalue > ALPHA
    print(f'Levene test: statistic = {lev.statistic:.4f}, p = {lev.pvalue:.4f} -> variances are '
          f'{"EQUAL (homogeneous)" if equal_var else "NOT equal"}')
    return tab, equal_var

ALT = {'two-sided': ('=', '!='), 'greater': ('<=', '>'), 'less': ('>=', '<')}
def t_critical(dof):
    return stats.t.ppf(1 - ALPHA/2, dof) if ALTERNATIVE == 'two-sided' else stats.t.ppf(1 - ALPHA, dof)

## Step 4: One-Sample t-test
Tests whether the **population mean** equals a given value $\mu_0$. $t = \dfrac{\bar{x}-\mu_0}{s/\sqrt{n}}$, df = n - 1.

In [ ]:
if s_t1_col is None:
    print('No numeric column available - skipped')
else:
    x = df[s_t1_col].dropna().astype(float)
    n, mean, sd = len(x), x.mean(), x.std(ddof=1)
    se = sd / np.sqrt(n)
    MU = MU0 if MU0 is not None else round(float(x.median()), 2)
    if MU0 is None:
        print(f'NOTE: MU0 not set -> using the sample median ({MU}) just for demonstration. Set MU0 in CONFIG to the value from your question.\n')

    print(f'Column: {s_t1_col} | n = {n} | sample mean = {mean:.4f} | sample std = {sd:.4f} | std error = {se:.4f}')
    print(f'Shapiro-Wilk normality p = {shapiro_p(x):.4f}  (n>30 -> t-test is fairly robust by CLT)')

    t_stat, p = stats.ttest_1samp(x, MU, alternative=ALTERNATIVE)
    h0 = f'mu {ALT[ALTERNATIVE][0]} {MU}'
    h1 = f'mu {ALT[ALTERNATIVE][1]} {MU}'
    decide('One-sample t-test', f'{s_t1_col} vs mu0={MU}', f't ({n-1} df)', t_stat, p, h0, h1)

    tc = stats.t.ppf(1 - ALPHA/2, n - 1)
    print(f'Critical t value ({ALTERNATIVE}) = {"+/-" if ALTERNATIVE=="two-sided" else ""}{t_critical(n-1):.4f}')
    print(f'{(1-ALPHA)*100:.0f}% confidence interval for the mean: ({mean - tc*se:.4f}, {mean + tc*se:.4f})')
    print(f"Cohen's d (effect size) = {(mean - MU)/sd:.3f}")

    plt.figure(figsize=(7, 4))
    sns.histplot(x, kde=True, color='steelblue')
    plt.axvline(mean, color='red', ls='--', label=f'sample mean = {mean:.2f}')
    plt.axvline(MU, color='green', ls='-', label=f'mu0 = {MU}')
    plt.title(f'One-sample t-test: {s_t1_col}'); plt.legend()
    plt.show()

## Step 5: Independent Two-Sample t-test
Compares the **means of two independent groups**. Levene's test decides: equal variances -> Student's t-test, unequal -> **Welch's** t-test.

In [ ]:
ok_t2 = False
if s_t2_num is None or s_t2_grp is None:
    print('Need one numeric and one categorical column - skipped')
else:
    d2 = df[[s_t2_num, s_t2_grp]].dropna().copy()
    d2[s_t2_grp] = d2[s_t2_grp].astype(str)
    if T2_LEVELS is not None:
        g1, g2 = str(T2_LEVELS[0]), str(T2_LEVELS[1])
    else:
        g1, g2 = d2[s_t2_grp].value_counts().index[:2]          # two largest groups
        if d2[s_t2_grp].nunique() > 2:
            print(f'NOTE: {s_t2_grp} has {d2[s_t2_grp].nunique()} levels -> comparing the two largest: {g1} vs {g2}. Use T2_LEVELS to choose.')
    a = d2.loc[d2[s_t2_grp] == g1, s_t2_num].astype(float).values
    b = d2.loc[d2[s_t2_grp] == g2, s_t2_num].astype(float).values
    ok_t2 = len(a) >= 2 and len(b) >= 2
    if not ok_t2:
        print('Not enough observations in the groups - skipped')

if ok_t2:
    print(f'Comparing {s_t2_num} between {s_t2_grp} = "{g1}" (group 1) and "{g2}" (group 2)\n')
    tab, equal_var = check_assumptions({g1: a, g2: b})
    print('->', "Using Student's t-test (equal variances)" if equal_var else "Using Welch's t-test (unequal variances)")

    t_stat, p = stats.ttest_ind(a, b, equal_var=equal_var, alternative=ALTERNATIVE)
    n1, n2, m1, m2, v1, v2 = len(a), len(b), a.mean(), b.mean(), a.var(ddof=1), b.var(ddof=1)
    sp = np.sqrt(((n1-1)*v1 + (n2-1)*v2) / (n1+n2-2))
    if equal_var:
        se, dof = sp*np.sqrt(1/n1 + 1/n2), n1 + n2 - 2
    else:
        se  = np.sqrt(v1/n1 + v2/n2)
        dof = (v1/n1 + v2/n2)**2 / ((v1/n1)**2/(n1-1) + (v2/n2)**2/(n2-1))

    h0 = f'mean({g1}) {ALT[ALTERNATIVE][0]} mean({g2})'
    h1 = f'mean({g1}) {ALT[ALTERNATIVE][1]} mean({g2})'
    decide('Independent t-test' + (' (Student)' if equal_var else ' (Welch)'), f'{s_t2_num} by {s_t2_grp}',
           f't ({dof:.1f} df)', t_stat, p, h0, h1)

    tc = stats.t.ppf(1 - ALPHA/2, dof)
    diff = m1 - m2
    print(f'Mean({g1}) = {m1:.4f}, Mean({g2}) = {m2:.4f}, difference = {diff:.4f}')
    print(f'Critical t value ({ALTERNATIVE}) = {"+/-" if ALTERNATIVE=="two-sided" else ""}{t_critical(dof):.4f}')
    print(f'{(1-ALPHA)*100:.0f}% CI for difference of means: ({diff - tc*se:.4f}, {diff + tc*se:.4f})')
    d_val = diff / sp
    size = 'small' if abs(d_val) < 0.5 else 'medium' if abs(d_val) < 0.8 else 'large'
    print(f"Cohen's d = {d_val:.3f}  ({'negligible' if abs(d_val) < 0.2 else size} effect)")

    # Non-parametric alternative (use if data are clearly non-normal and samples are small)
    u, p_u = stats.mannwhitneyu(a, b, alternative=ALTERNATIVE)
    print(f'\n[Non-parametric alternative] Mann-Whitney U = {u:.2f}, p = {p_u:.6f} -> '
          f'{"Reject H0" if p_u <= ALPHA else "Fail to reject H0"}')

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    plot_d = d2[d2[s_t2_grp].isin([g1, g2])]
    sns.boxplot(x=s_t2_grp, y=s_t2_num, data=plot_d, order=[g1, g2], showmeans=True,
                meanprops={'marker': 'D', 'markerfacecolor': 'red', 'markeredgecolor': 'red'}, ax=axes[0], palette='Set2')
    axes[0].set_title('Boxplot (red diamond = mean)')
    sns.histplot(data=plot_d, x=s_t2_num, hue=s_t2_grp, kde=True, element='step', ax=axes[1])
    axes[1].set_title('Distributions')
    plt.tight_layout()
    plt.show()

## Step 6: Paired t-test
Use when the **same subjects are measured twice** (before/after). Tests whether the mean difference is 0. Applies a one-sample t-test on the differences $d = x_1 - x_2$.
> Use this only if your two columns are truly paired measurements - otherwise skip.

In [ ]:
if s_pair1 is None or s_pair2 is None or s_pair1 == s_pair2:
    print('Need two different numeric columns - skipped')
else:
    pr = df[[s_pair1, s_pair2]].dropna().astype(float)
    x1, x2 = pr[s_pair1].values, pr[s_pair2].values
    diffs = x1 - x2
    n, md_, sd_ = len(diffs), diffs.mean(), diffs.std(ddof=1)
    print(f'Pairs: {s_pair1} (first) vs {s_pair2} (second) | n = {n}')
    print(f'Mean difference ({s_pair1} - {s_pair2}) = {md_:.4f} | std of differences = {sd_:.4f}')
    print(f'Shapiro-Wilk on differences: p = {shapiro_p(diffs):.4f}')

    t_stat, p = stats.ttest_rel(x1, x2, alternative=ALTERNATIVE)
    h0 = f'mean difference {ALT[ALTERNATIVE][0]} 0'
    h1 = f'mean difference {ALT[ALTERNATIVE][1]} 0'
    decide('Paired t-test', f'{s_pair1} vs {s_pair2}', f't ({n-1} df)', t_stat, p, h0, h1)

    tc = stats.t.ppf(1 - ALPHA/2, n - 1)
    se = sd_ / np.sqrt(n)
    print(f'Critical t value ({ALTERNATIVE}) = {"+/-" if ALTERNATIVE=="two-sided" else ""}{t_critical(n-1):.4f}')
    print(f'{(1-ALPHA)*100:.0f}% CI for mean difference: ({md_ - tc*se:.4f}, {md_ + tc*se:.4f})')
    print(f"Cohen's d (paired) = {md_/sd_:.3f}")

    if np.any(diffs != 0):
        w, p_w = stats.wilcoxon(x1, x2, alternative=ALTERNATIVE)
        print(f'\n[Non-parametric alternative] Wilcoxon signed-rank W = {w:.2f}, p = {p_w:.6f} -> '
              f'{"Reject H0" if p_w <= ALPHA else "Fail to reject H0"}')

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(diffs, kde=True, ax=axes[0], color='teal')
    axes[0].axvline(0, color='k', ls='--'); axes[0].axvline(md_, color='red', label=f'mean diff = {md_:.2f}')
    axes[0].set_title('Differences (H0: centred at 0)'); axes[0].legend()
    sns.boxplot(data=pr, ax=axes[1], palette='Set2'); axes[1].set_title('Paired columns')
    plt.tight_layout()
    plt.show()

## Step 7: Chi-Square Test of Independence
Tests whether **two categorical variables are associated**. $\chi^2 = \sum \dfrac{(O-E)^2}{E}$, df = (rows-1)(cols-1). Needs expected counts >= 5 in most cells.

In [ ]:
if s_chi1 is None or s_chi2 is None or s_chi1 == s_chi2:
    print('Need two different categorical columns - skipped')
else:
    cd = df[[s_chi1, s_chi2]].dropna().astype(str)
    ct = pd.crosstab(cd[s_chi1], cd[s_chi2])
    print(f'Variables: {s_chi1} (rows) x {s_chi2} (columns)  | n = {ct.values.sum()}')
    print('\nObserved frequencies:')
    display(pd.crosstab(cd[s_chi1], cd[s_chi2], margins=True, margins_name='Total'))

    chi2, p, dof, expected = stats.chi2_contingency(ct)          # Yates correction applied automatically for 2x2
    exp_df = pd.DataFrame(expected, index=ct.index, columns=ct.columns).round(2)
    print('Expected frequencies (if independent):')
    display(exp_df)
    pct_low = (expected < 5).mean() * 100
    print(f'Cells with expected count < 5: {pct_low:.1f}%  ' + ('(OK: assumption satisfied)' if pct_low <= 20 else '(WARNING: assumption violated -> use Fisher exact test for 2x2 / merge categories)'))

    h0 = f'{s_chi1} and {s_chi2} are independent (no association)'
    h1 = f'{s_chi1} and {s_chi2} are associated (dependent)'
    decide('Chi-square independence', f'{s_chi1} x {s_chi2}', f'chi2 ({dof} df)', chi2, p, h0, h1)
    print(f'Critical chi2 value (alpha={ALPHA}, df={dof}) = {stats.chi2.ppf(1 - ALPHA, dof):.4f}')

    chi2_nc = stats.chi2_contingency(ct, correction=False)[0]
    n_tot = ct.values.sum()
    v = np.sqrt(chi2_nc / (n_tot * (min(ct.shape) - 1)))
    print(f"Cramer's V (effect size) = {v:.3f}  ({'negligible' if v < 0.1 else 'small' if v < 0.3 else 'medium' if v < 0.5 else 'large'} association)")

    if ct.shape == (2, 2):
        odds, p_f = stats.fisher_exact(ct)
        print(f'\n[2x2 alternative] Fisher exact test: odds ratio = {odds:.3f}, p = {p_f:.6f} -> '
              f'{"Reject H0" if p_f <= ALPHA else "Fail to reject H0"}')

    resid = pd.DataFrame((ct.values - expected) / np.sqrt(expected), index=ct.index, columns=ct.columns)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    sns.heatmap(ct, annot=True, fmt='d', cmap='Blues', ax=axes[0]); axes[0].set_title('Observed counts')
    sns.heatmap(resid, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=axes[1])
    axes[1].set_title('Standardised residuals (|value| > 2 = cell drives the association)')
    plt.tight_layout()
    plt.show()

    (ct.div(ct.sum(axis=1), axis=0) * 100).plot(kind='bar', stacked=True, figsize=(7, 4), colormap='Set2')
    plt.ylabel('% within ' + s_chi1); plt.title(f'{s_chi2} distribution within each {s_chi1}')
    plt.legend(title=s_chi2, bbox_to_anchor=(1.02, 1), loc='upper left'); plt.xticks(rotation=0)
    plt.show()

## Step 8: Chi-Square Goodness of Fit
Tests whether the **observed category counts match an expected distribution** (default: all categories equally likely).

In [ ]:
if s_gof is None:
    print('No categorical column - skipped')
else:
    obs = df[s_gof].dropna().astype(str).value_counts().sort_index()
    k, n_tot = len(obs), obs.sum()
    props = np.ones(k) / k if GOF_PROPS is None else np.array(GOF_PROPS, dtype=float) / np.sum(GOF_PROPS)
    if len(props) != k:
        print(f'GOF_PROPS must have {k} values (one per category in this order: {list(obs.index)}) - skipped')
    else:
        exp = props * n_tot
        table = pd.DataFrame({'Observed': obs.values, 'Expected': exp.round(2),
                              '(O-E)^2/E': ((obs.values - exp)**2 / exp).round(3)}, index=obs.index)
        display(table)
        print('All expected counts >= 5:', bool((exp >= 5).all()))

        chi2, p = stats.chisquare(obs.values, f_exp=exp)
        h0 = 'Observed distribution matches the expected distribution' + (' (all categories equally likely)' if GOF_PROPS is None else '')
        h1 = 'Observed distribution differs from the expected distribution'
        decide('Chi-square goodness of fit', s_gof, f'chi2 ({k-1} df)', chi2, p, h0, h1)
        print(f'Critical chi2 value (alpha={ALPHA}, df={k-1}) = {stats.chi2.ppf(1 - ALPHA, k - 1):.4f}')

        x = np.arange(k)
        plt.figure(figsize=(7, 4))
        plt.bar(x - 0.2, obs.values, width=0.4, label='Observed', color='steelblue')
        plt.bar(x + 0.2, exp, width=0.4, label='Expected', color='orange')
        plt.xticks(x, obs.index, rotation=30); plt.title(f'Goodness of fit: {s_gof}'); plt.legend()
        plt.show()

## Step 9: One-Way ANOVA
Compares the **means of 3 or more groups**. $F = \dfrac{MS_{between}}{MS_{within}}$. If significant, **Tukey's HSD** shows *which* groups differ.
Assumptions: independent observations, normality within groups, equal variances (Levene).

In [ ]:
ok_an = False
if s_an_num is None or s_an_grp is None:
    print('Need one numeric and one categorical column - skipped')
else:
    da = df[[s_an_num, s_an_grp]].dropna().copy()
    da[s_an_grp] = da[s_an_grp].astype(str)
    da[s_an_num] = da[s_an_num].astype(float)
    counts = da[s_an_grp].value_counts()
    keep = counts[counts >= 2].index.tolist()
    da = da[da[s_an_grp].isin(keep)]
    groups = {g: da.loc[da[s_an_grp] == g, s_an_num].values for g in sorted(keep)}
    ok_an = len(groups) >= 2
    if len(groups) > 15:
        print(f'WARNING: {s_an_grp} has {len(groups)} groups - choose a column with fewer groups.')
    if len(groups) == 2:
        print('NOTE: only 2 groups -> ANOVA gives the same result as the two-sample t-test (F = t^2).')
    if not ok_an:
        print('Need at least 2 groups with 2+ observations - skipped')

if ok_an:
    print(f'{s_an_num} compared across {len(groups)} groups of {s_an_grp}\n')
    tab, equal_var = check_assumptions(groups)

    F, p = stats.f_oneway(*groups.values())
    k, N = len(groups), len(da)
    grand = da[s_an_num].mean()
    ss_b = sum(len(v) * (v.mean() - grand)**2 for v in groups.values())
    ss_w = sum(((v - v.mean())**2).sum() for v in groups.values())
    df_b, df_w = k - 1, N - k
    ms_b, ms_w = ss_b / df_b, ss_w / df_w

    print('\nANOVA table:')
    anova_tab = pd.DataFrame({'Sum of Squares': [ss_b, ss_w, ss_b + ss_w],
                              'df': [df_b, df_w, N - 1],
                              'Mean Square': [ms_b, ms_w, np.nan],
                              'F': [F, np.nan, np.nan],
                              'p-value': [p, np.nan, np.nan]},
                             index=['Between groups', 'Within groups', 'Total']).round(4)
    display(anova_tab)

    h0 = 'All group means are equal (mu1 = mu2 = ... = muk)'
    h1 = 'At least one group mean is different'
    decide('One-way ANOVA', f'{s_an_num} by {s_an_grp}', f'F ({df_b}, {df_w} df)', F, p, h0, h1)
    print(f'Critical F value (alpha={ALPHA}) = {stats.f.ppf(1 - ALPHA, df_b, df_w):.4f}')
    eta = ss_b / (ss_b + ss_w)
    print(f'Eta-squared (effect size) = {eta:.3f}  ({eta*100:.1f}% of variance explained by group; '
          f'{"small" if eta < 0.06 else "medium" if eta < 0.14 else "large"} effect)')

    if not equal_var:
        print('\nNOTE: Levene rejected equal variances -> classic ANOVA is less reliable; rely on Kruskal-Wallis / Welch-type results too.')
    H, p_k = stats.kruskal(*groups.values())
    print(f'\n[Non-parametric alternative] Kruskal-Wallis H = {H:.3f}, p = {p_k:.6f} -> '
          f'{"Reject H0" if p_k <= ALPHA else "Fail to reject H0"}')

In [ ]:
# Post-hoc: Tukey HSD (only meaningful when ANOVA is significant and there are 3+ groups)
if ok_an and len(groups) >= 3:
    if p <= ALPHA:
        print('ANOVA is significant -> Tukey HSD pairwise comparisons:\n')
        try:
            from statsmodels.stats.multicomp import pairwise_tukeyhsd
            tuk = pairwise_tukeyhsd(da[s_an_num], da[s_an_grp], alpha=ALPHA)
            print(tuk.summary())
            tuk.plot_simultaneous(figsize=(7, 4)); plt.show()
        except ImportError:
            res = stats.tukey_hsd(*groups.values())
            names = list(groups.keys())
            display(pd.DataFrame(res.pvalue, index=names, columns=names).round(4))
            print('(table shows pairwise p-values; p <= alpha means the two groups differ)')
    else:
        print('ANOVA not significant -> post-hoc test not needed.')

if ok_an:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    order = list(groups.keys())
    sns.boxplot(x=s_an_grp, y=s_an_num, data=da, order=order, showmeans=True,
                meanprops={'marker': 'D', 'markerfacecolor': 'red', 'markeredgecolor': 'red'}, ax=axes[0], palette='Set2')
    axes[0].set_title('Boxplot by group (red diamond = mean)'); axes[0].tick_params(axis='x', rotation=30)
    sns.pointplot(x=s_an_grp, y=s_an_num, data=da, order=order, errorbar=('ci', 95), capsize=0.15, ax=axes[1])
    axes[1].set_title('Group means with 95% CI'); axes[1].tick_params(axis='x', rotation=30)
    plt.tight_layout()
    plt.show()

## Step 10: Summary of All Tests Performed

In [ ]:
summary_df = pd.DataFrame(summary)
display(summary_df)
summary_df.to_csv('hypothesis_test_results.csv', index=False)
print('Saved:', os.path.join(os.getcwd(), 'hypothesis_test_results.csv'))

## Theory Cheat-Sheet (for viva / written answers)

**Steps of any hypothesis test:** (1) state H0 and H1 -> (2) choose alpha (usually 0.05) -> (3) choose test & compute test statistic -> (4) get p-value / critical value -> (5) decide -> (6) conclude in words.

**Decision rule:** if **p <= alpha -> reject H0**; if p > alpha -> fail to reject H0 (we never "accept" H0). Equivalent: reject H0 if |test statistic| > critical value.

**p-value:** probability of getting a result at least as extreme as the observed one **assuming H0 is true**. It is *not* the probability that H0 is true.

| Error | Meaning | Probability |
|---|---|---|
| Type I | Reject H0 when it is true (false positive) | alpha |
| Type II | Fail to reject H0 when it is false (false negative) | beta (power = 1 - beta) |

| Situation | Test | Non-parametric alternative |
|---|---|---|
| Mean of one sample vs a known value | One-sample t-test | Wilcoxon signed-rank |
| Means of 2 independent groups | Independent t-test (Student if equal variances, **Welch** if not) | Mann-Whitney U |
| Means of the same subjects measured twice | Paired t-test | Wilcoxon signed-rank |
| Means of 3+ groups | One-way ANOVA (+ Tukey HSD if significant) | Kruskal-Wallis |
| Association between 2 categorical variables | Chi-square test of independence | Fisher exact (2x2, small counts) |
| Categorical counts vs expected proportions | Chi-square goodness of fit | Exact multinomial test |

| Assumption check | Test | Meaning of p <= alpha |
|---|---|---|
| Normality | Shapiro-Wilk | data are NOT normal |
| Equal variances | Levene | variances are NOT equal |

**Effect sizes:** Cohen's d (0.2 small, 0.5 medium, 0.8 large) | Eta-squared (0.01 small, 0.06 medium, 0.14 large) | Cramer's V (0.1 small, 0.3 medium, 0.5 large). A tiny p-value with a tiny effect size can still be practically unimportant.

**Why not many t-tests instead of ANOVA?** Each extra test inflates the overall Type I error (family-wise error rate); ANOVA tests all groups at once, then post-hoc tests correct for multiple comparisons.